We benchmarked 7 methods on a fair medical resource allocation task at two fairness levels (α=0.5, α=2.0). Each method trains a neural predictor whose output feeds an α-fair allocation solver; we measure decision regret (welfare lost vs. oracle), prediction MSE, and demographic fairness (equalized group MSE). We normalize regret by the optimal welfare so results are comparable across α values.

Key finding: Multi-objective optimization methods — particularly PCGrad (5.0% welfare loss at α=0.5, 13.1% at α=2.0) — consistently outperform both the predict-then-optimize baseline FPTO and the naive decision-focused baseline FDFL. The gradient conflict analysis shows that decision and fairness gradients frequently oppose each other (negative cosine similarity), which explains why MOO methods that explicitly handle conflicts (PCGrad, MGDA) beat simple weighted sums.

FFO (differentiable CVXPY layer) is unstable at α=2.0 (92% welfare loss) and needs further debugging. All other methods use fast closed-form analytical gradients via implicit differentiation of the KKT conditions, so training is fast (~seconds per run on GPU).



In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Fair Decision-Focused Learning for Medical Resource Allocation

This notebook runs the full fair DFL medical resource allocation experiment.
It is designed to run locally via VSCode connected to a Colab runtime (or any local Python environment).

---
## 1. Problem Formulation

### 1.1 Overview

We study **fair resource allocation in healthcare** using **decision-focused learning (DFL)**.

The pipeline has two stages:
1. A **prediction model** $\hat{r} = f_\theta(x)$ estimates patient benefit from features $x$.
2. An **optimization layer** $d^*(\hat{r})$ allocates scarce medical resources given the predicted benefits.

The central challenge is to **simultaneously balance** three competing objectives:
- **Decision quality** (regret): how much utility we lose by acting on predictions instead of true benefits.
- **Prediction accuracy**: how close our predictions are to the truth (MSE).
- **Demographic fairness**: whether prediction errors are equitable across racial groups.

### 1.2 The Allocation Problem

Given $n$ patients, each with:
- Predicted benefit $\hat{r}_i$
- Cost $c_i$
- Demographic group $g_i$ (race)

We solve the **$\alpha$-fair resource allocation** problem:

$$\max_{d} \sum_i f_\alpha(\hat{r}_i \cdot d_i) \quad \text{s.t.} \quad \sum_i c_i \cdot d_i \le Q, \quad d_i \ge 0$$

where the **$\alpha$-fairness utility** function is:

$$f_\alpha(u) = \begin{cases} \frac{u^{1-\alpha}}{1-\alpha} & \alpha \ne 1 \\ \log(u) & \alpha = 1 \end{cases}$$

The parameter $\alpha$ controls the fairness-efficiency tradeoff:
- $\alpha = 0$: **Utilitarian** (maximize total utility)
- $\alpha = 1$: **Proportional fairness** (Nash bargaining)
- $\alpha \to \infty$: **Max-min (Rawlsian)** fairness

**Group-coupled version:** We apply an outer $\alpha$-fairness objective over group utilities $\mu_k = F_\beta(\text{within-group utilities})$, where $\beta = \alpha$ creates a two-level fairness structure. Each group's aggregate utility is computed using $\alpha$-fair aggregation within the group, and then outer $\alpha$-fair aggregation is applied across groups.

### 1.3 Three Objectives

We optimize three objectives simultaneously:

1. **Decision Regret** $\mathcal{L}_{\text{dec}}$:
$$\mathcal{L}_{\text{dec}} = \text{Obj}(d^*(r_{\text{true}})) - \text{Obj}(d^*(\hat{r}))$$
This measures how much social welfare we lose by using predicted benefits $\hat{r}$ instead of true benefits $r_{\text{true}}$ in the optimization.

2. **Prediction Loss** $\mathcal{L}_{\text{pred}}$:
$$\mathcal{L}_{\text{pred}} = \text{MSE}(\hat{r},\, r_{\text{true}}) = \frac{1}{n}\sum_i (\hat{r}_i - r_i)^2$$

3. **Prediction Fairness** $\mathcal{L}_{\text{fair}}$:
$$\mathcal{L}_{\text{fair}} = \text{MAD}\bigl(\{\text{MSE}_k\}_{k=1}^K\bigr)$$
The **Mean Absolute Deviation** of per-group MSE values from their grand mean. This penalizes disparities in prediction quality across demographic groups.

### 1.4 Methods

#### Non-MOO Methods
These methods use at most two of the three objectives:

| Method | Description | Objectives Used |
|--------|-------------|----------------|
| **FPTO** (Fair Predict-Then-Optimize) | Train predictor with $\mathcal{L}_{\text{pred}} + \lambda \cdot \mathcal{L}_{\text{fair}}$; no decision gradient | pred + fair |
| **FDFL** (Fair Decision-Focused Learning) | Train with $\mathcal{L}_{\text{dec}} + \lambda \cdot \mathcal{L}_{\text{fair}}$; decision gradient via closed-form Jacobian | dec + fair |
| **FFO** (Fair Fold-Opt) | Differentiable optimization layer via cvxpylayers; replaces closed-form decision gradient | dec + pred + fair |
| **LANCER** | Learned surrogate neural network for the decision gradient | dec + pred + fair |

#### MOO Methods (all 3 objectives active simultaneously)
These methods use multi-objective optimization to handle all three objectives with individual per-objective gradients:

| Method | Description |
|--------|-------------|
| **Weighted Sum** (4 configs) | $w_{\text{dec}} \cdot g_{\text{dec}} + w_{\text{pred}} \cdot g_{\text{pred}} + w_{\text{fair}} \cdot g_{\text{fair}}$ |
| **MGDA** | Min-norm point in convex hull of per-objective gradients (Sener & Koltun, 2018) |
| **PCGrad** | Project away conflicting gradient components (Yu et al., 2020) |
| **CAGrad** | Conflict-averse gradient descent with parameter $c$ (Liu et al., ICLR 2021) |
| **FAMO** | Fast Adaptive Multitask Optimization with dynamic softmax weights (Liu et al., NeurIPS 2023) |
| **PLG-FP** | Nested PLG: fairness-primary in pred-space, then decision-primary final |
| **PLG-PP** | Nested PLG: prediction-primary in pred-space, then decision-primary final |

### 1.5 Dataset

We use a healthcare utilization study dataset with:
- **48,784 patients** total
- **170 features**: demographics, comorbidities, costs, lab values, medications
- **Protected attribute**: race (0 = white, $n$ = 43,202; 1 = black, $n$ = 5,582)

**Benefit construction:**
$$\text{benefit} = 0.5 \cdot \log(1 + \text{avoidable\_cost})_{\text{norm}} + 0.5 \cdot \frac{\text{gagne\_sum\_t}}{17}$$

**Cost:** total cost capped at the 99th percentile, normalized to $[0, 1]$.

**Key disparity:** Race=1 (Black) patients have 46% higher mean benefit but 12.7% higher mean cost, reflecting the well-documented phenomenon where algorithmic predictions can systematically under-serve minority groups.

### 1.6 Fairness Measure: MAD

We use the **Mean Absolute Deviation (MAD)** of per-group MSE from the grand mean:

$$\text{MAD} = \frac{1}{K} \sum_{k=1}^{K} \left| \text{MSE}_k - \overline{\text{MSE}} \right|$$

where $\text{MSE}_k = \frac{1}{n_k} \sum_{i \in \text{group}_k} (\hat{r}_i - r_i)^2$ and $\overline{\text{MSE}} = \frac{1}{K} \sum_k \text{MSE}_k$.

For differentiability, we use a **smoothed** version:

$$\text{MAD}_{\text{smooth}} = \frac{1}{K} \sum_{k=1}^{K} \sqrt{(\text{MSE}_k - \overline{\text{MSE}})^2 + \varepsilon}$$

with $\varepsilon = 10^{-6}$ to ensure a well-defined gradient everywhere.

---
## 2. Setup

Install dependencies and prepare the source code.

In [1]:
# Install dependencies
!pip install -q torch numpy pandas scipy matplotlib cvxpy mosek
!pip install -q git+https://github.com/cvxpy/cvxtorch.git
!pip install -q --upgrade threadpoolctl


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import sys

# ============================================================
# COLAB PATH SETUP
# Upload the colab_upload/ folder to Google Drive as:
#   My Drive/fdfl_experiment/
# Then this cell sets all paths automatically.
# ============================================================

DRIVE_ROOT     = '/content/drive/MyDrive/fdfl_experiment'
DATA_CSV_PATH  = f'{DRIVE_ROOT}/data/data_processed.csv'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'
RESULTS_DIR    = f'{DRIVE_ROOT}/results'

# Set MOSEK license path
if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

# Add the package source to sys.path
SRC_DIR = os.path.join(DRIVE_ROOT, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'DRIVE_ROOT:    {DRIVE_ROOT}')
print(f'DATA_CSV_PATH: {DATA_CSV_PATH}')
print(f'SRC_DIR:       {SRC_DIR}')
print(f'RESULTS_DIR:   {RESULTS_DIR}')


In [11]:
# Verify that the fair_dfl package loads correctly.
# The pyepo try/except guard and runner.py annotations fix have been applied
# directly in the source files.
try:
    from fair_dfl.tasks import MedicalResourceAllocationTask
    print('OK: MedicalResourceAllocationTask imported successfully')
except ImportError as e:
    print(f'Import error: {e}')
    print('Make sure the source files at SRC_DIR are accessible.')
    raise


Import error: No module named 'fair_dfl'
Make sure the source files at SRC_DIR are accessible.


ModuleNotFoundError: No module named 'fair_dfl'

In [ ]:
# Verify runner.py imports correctly
try:
    from fair_dfl.runner import run_experiment_unified
    print('OK: run_experiment_unified imported successfully')
except ImportError as e:
    print(f'Import error: {e}')
    raise


In [ ]:
# Verify imports work
import torch
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt

from fair_dfl.runner import run_experiment_unified
from fair_dfl.tasks.medical_resource_allocation import MedicalResourceAllocationTask

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')


---
## 3. Data Loading & Statistics

In [ ]:
# Load and inspect the dataset
assert os.path.exists(DATA_CSV_PATH), (
    f"Data file not found at {DATA_CSV_PATH}. "
    "Please upload data_processed.csv or update DATA_CSV_PATH."
)

df = pd.read_csv(DATA_CSV_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Number of patients: {len(df):,}")
print(f"Number of features: {df.shape[1]}")
print()
print("Columns (first 30):")
print(list(df.columns[:30]))
print(f"... and {df.shape[1] - 30} more columns")

In [ ]:
# Race distribution
print("=" * 50)
print("Race Distribution")
print("=" * 50)
race_counts = df['race'].value_counts().sort_index()
for race_val, count in race_counts.items():
    label = 'White' if race_val == 0 else 'Black'
    print(f"  Race={race_val} ({label}): n={count:,} ({100*count/len(df):.1f}%)")
print()

In [ ]:
# Benefit and cost distributions by race
print("=" * 50)
print("Benefit Distribution by Race")
print("=" * 50)
for race_val in sorted(df['race'].unique()):
    label = 'White' if race_val == 0 else 'Black'
    subset = df[df['race'] == race_val]
    b = subset['benefit']
    print(f"  Race={race_val} ({label}): mean={b.mean():.4f}, "
          f"std={b.std():.4f}, min={b.min():.4f}, max={b.max():.4f}")

# Compute disparity ratio
mean_0 = df[df['race'] == 0]['benefit'].mean()
mean_1 = df[df['race'] == 1]['benefit'].mean()
print(f"\n  Benefit ratio (Black/White): {mean_1/mean_0:.3f} "
      f"({100*(mean_1/mean_0 - 1):.1f}% higher for Black patients)")

print()
print("=" * 50)
print("Cost Distribution by Race")
print("=" * 50)
for race_val in sorted(df['race'].unique()):
    label = 'White' if race_val == 0 else 'Black'
    subset = df[df['race'] == race_val]
    c = subset['cost_t_capped']
    print(f"  Race={race_val} ({label}): mean={c.mean():.4f}, "
          f"std={c.std():.4f}")

cost_0 = df[df['race'] == 0]['cost_t_capped'].mean()
cost_1 = df[df['race'] == 1]['cost_t_capped'].mean()
print(f"\n  Cost ratio (Black/White): {cost_1/cost_0:.3f} "
      f"({100*(cost_1/cost_0 - 1):.1f}% higher for Black patients)")

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Benefit distribution
ax = axes[0]
for race_val, color, label in [(0, '#4C72B0', 'White (race=0)'), (1, '#DD8452', 'Black (race=1)')]:
    subset = df[df['race'] == race_val]['benefit']
    ax.hist(subset, bins=50, alpha=0.6, color=color, label=label, density=True)
ax.set_xlabel('Benefit')
ax.set_ylabel('Density')
ax.set_title('Benefit Distribution by Race')
ax.legend()

# Cost distribution
ax = axes[1]
for race_val, color, label in [(0, '#4C72B0', 'White (race=0)'), (1, '#DD8452', 'Black (race=1)')]:
    subset = df[df['race'] == race_val]['cost_t_capped']
    ax.hist(subset, bins=50, alpha=0.6, color=color, label=label, density=True)
ax.set_xlabel('Cost (capped)')
ax.set_ylabel('Density')
ax.set_title('Cost Distribution by Race')
ax.legend()

plt.tight_layout()
plt.show()

---
## 4. Experiment Configuration

Define the experiment grid covering all 14 methods, 2 alpha values, 4 lambda values, and 3 seeds.

In [ ]:
# ========================================================================
# Experiment Configuration
# ========================================================================

# Sample size: set to 500 for quick test, 0 for full dataset (48,784)
N_SAMPLE_SMALL = 500   # For Section 5 (quick verification)
N_SAMPLE_FULL = 0      # For Section 6 (0 = use all patients)

# Alpha-fairness parameters to sweep
ALPHA_VALUES = [0.5, 2.0]

# Base task configuration
def make_task_cfg(n_sample, alpha_fair):
    """Create a task configuration dictionary."""
    return {
        "name": "medical_resource_allocation",
        "data_csv": DATA_CSV_PATH,
        "n_sample": n_sample,
        "data_seed": 42,
        "split_seed": 2,
        "test_fraction": 0.5,
        "val_fraction": 0.2,
        "alpha_fair": alpha_fair,
        "budget": -1,           # Dynamic from rho
        "budget_rho": 0.35,
        "decision_mode": "group",
        "fairness_type": "mad",
    }

# Base training configuration
base_train_cfg = {
    "lambdas": [0.0, 0.05, 0.2, 0.5],
    "seeds": [11, 22, 33],
    "steps_per_lambda": 70,
    "batch_size": -1,  # Will be set to full training set size (see below)
    "lr": 0.0005,
    "lr_decay": 0.0005,
    "alpha_schedule": {"type": "inv_sqrt", "alpha0": 1.0, "alpha_min": 0.0},
    "force_lambda_path_all_methods": True,
    "grad_clip_norm": 10000.0,
    "explode_threshold": 1000000.0,
    "fairness_smoothing": 1e-6,
    "log_every": 5,
    "pareto_sweep_mode": True,
    "lambda_train": 0.0,
    "model": {
        "arch": "mlp",
        "hidden_dim": 64,
        "n_layers": 2,
        "activation": "relu",
        "dropout": 0.0,
        "batch_norm": False,
        "init_mode": "default",
    },
    "device": DEVICE,
}

print("Base training configuration:")
for k, v in base_train_cfg.items():
    print(f"  {k}: {v}")


In [ ]:
# ========================================================================
# Method Configurations (unified inline-flags format)
# ========================================================================

method_configs = {
    # ----- Non-MOO Methods -----
    "FPTO":     {"method": "fpto",  "use_dec": False, "use_pred": True,  "use_fair": True,
                 "pred_weight_mode": "fixed1"},
    "FDFL":     {"method": "fdfl",  "use_dec": True,  "use_pred": False, "use_fair": True,
                 "pred_weight_mode": "zero"},
    "FFO":      {"method": "ffo",   "use_dec": True,  "use_pred": True,  "use_fair": True,
                 "pred_weight_mode": "schedule", "continuation": True,
                 "allow_orthogonalization": True, "decision_grad_backend": "ffo"},

    # ----- MOO Methods -----
    "WS-equal": {"method": "fplg", "use_dec": True, "use_pred": True, "use_fair": True,
                 "pred_weight_mode": "schedule", "continuation": True,
                 "allow_orthogonalization": True,
                 "mo_method": "weighted_sum",
                 "mo_weights": {"decision_regret": 0.333, "pred_loss": 0.333, "pred_fairness": 0.333}},
    "WS-dec":   {"method": "fplg", "use_dec": True, "use_pred": True, "use_fair": True,
                 "pred_weight_mode": "schedule", "continuation": True,
                 "allow_orthogonalization": True,
                 "mo_method": "weighted_sum",
                 "mo_weights": {"decision_regret": 0.6, "pred_loss": 0.2, "pred_fairness": 0.2}},
    "MGDA":     {"method": "fplg", "use_dec": True, "use_pred": True, "use_fair": True,
                 "pred_weight_mode": "schedule", "continuation": True,
                 "allow_orthogonalization": True, "mo_method": "mgda"},
    "PCGrad":   {"method": "fplg", "use_dec": True, "use_pred": True, "use_fair": True,
                 "pred_weight_mode": "schedule", "continuation": True,
                 "allow_orthogonalization": True, "mo_method": "pcgrad"},
    "PLG-FP":   {"method": "fplg", "use_dec": True, "use_pred": True, "use_fair": True,
                 "pred_weight_mode": "schedule", "continuation": True,
                 "allow_orthogonalization": True, "mo_method": "plg_fp"},
    "PLG-PP":   {"method": "fplg", "use_dec": True, "use_pred": True, "use_fair": True,
                 "pred_weight_mode": "schedule", "continuation": True,
                 "allow_orthogonalization": True, "mo_method": "plg_pp"},
}

print(f"Total method configurations: {len(method_configs)}")
print()
for i, (name, spec) in enumerate(method_configs.items()):
    objectives = []
    if spec.get("use_dec"): objectives.append("dec")
    if spec.get("use_pred"): objectives.append("pred")
    if spec.get("use_fair"): objectives.append("fair")
    obj_str = "+".join(objectives)
    mo = spec.get("mo_method", "")
    extra = f", mo={mo}" if mo else ""
    dgb = spec.get("decision_grad_backend", "")
    if dgb: extra += f", dec_grad={dgb}"
    print(f"  [{i+1:2d}] {name:15s} objectives={obj_str}{extra}")


In [ ]:
# ========================================================================
# Helper: compute batch size from data
# ========================================================================

def compute_full_batch_size(data_csv, n_sample, test_fraction=0.5, val_fraction=0.2):
    """Compute the full training set size for use as batch_size.

    We use full-batch training because mini-batch breaks the budget constraint:
    the allocation solver needs to see all patients simultaneously to respect
    the global budget Q.
    """
    df_tmp = pd.read_csv(data_csv)
    n_total = n_sample if (n_sample > 0 and n_sample < len(df_tmp)) else len(df_tmp)
    n_test = int(round(test_fraction * n_total))
    n_remaining = n_total - n_test
    n_val = int(round(val_fraction * n_remaining))
    n_train = n_remaining - n_val
    return n_train

# Show batch sizes for both experiment modes
bs_small = compute_full_batch_size(DATA_CSV_PATH, N_SAMPLE_SMALL)
bs_full = compute_full_batch_size(DATA_CSV_PATH, N_SAMPLE_FULL)
print(f"Small experiment (n={N_SAMPLE_SMALL}): train batch_size = {bs_small}")
print(f"Full experiment (n={N_SAMPLE_FULL} = all): train batch_size = {bs_full}")

In [ ]:
# ========================================================================
# Core experiment runner function (unified pipeline)
# ========================================================================

import copy
import time
import traceback

def run_full_experiment(n_sample, alpha_values, method_configs, base_train_cfg,
                        data_csv_path=DATA_CSV_PATH, verbose=True):
    """
    Run the complete experiment grid using the unified training pipeline.

    For each alpha value, runs all methods via run_experiment_unified().

    Parameters
    ----------
    n_sample : int
        Number of patients to sample (0 = use all).
    alpha_values : list of float
        Alpha-fairness values to sweep.
    method_configs : dict
        Method configurations dict {name: config} with inline flags.
    base_train_cfg : dict
        Base training configuration.
    data_csv_path : str
        Path to data_processed.csv.
    verbose : bool
        Print progress.

    Returns
    -------
    all_stage_results : pd.DataFrame
    all_iter_results : pd.DataFrame
    errors : list of dict
    """
    batch_size = compute_full_batch_size(data_csv_path, n_sample)

    # Build mapping from lowercase method names to display names
    name_map = {k.lower(): k for k in method_configs}

    all_stage_dfs = []
    all_iter_dfs = []
    errors = []

    total_runs = len(alpha_values)

    for run_idx, alpha in enumerate(alpha_values, 1):
        task_cfg = make_task_cfg(n_sample=n_sample, alpha_fair=alpha)

        if verbose:
            print(f"\n[{run_idx}/{total_runs}] alpha={alpha}, "
                  f"{len(method_configs)} methods")

        train_cfg = copy.deepcopy(base_train_cfg)
        train_cfg["batch_size"] = batch_size
        cfg = {"task": task_cfg, "training": train_cfg}

        try:
            t0 = time.time()
            stage_df, iter_df = run_experiment_unified(
                cfg, method_configs=method_configs,
            )
            elapsed = time.time() - t0

            # Map lowercase 'method' column to display names in 'config_name'
            if "method" in stage_df.columns:
                stage_df["config_name"] = stage_df["method"].map(name_map)
            if "method" in iter_df.columns:
                iter_df["config_name"] = iter_df["method"].map(name_map)
            stage_df["alpha_fair"] = alpha
            iter_df["alpha_fair"] = alpha

            all_stage_dfs.append(stage_df)
            all_iter_dfs.append(iter_df)

            if verbose:
                n_rows = len(stage_df)
                if n_rows > 0 and "test_regret" in stage_df.columns:
                    print(f"    Done in {elapsed:.1f}s | {n_rows} stage rows | "
                          f"mean regret={stage_df['test_regret'].mean():.4f}")
                else:
                    print(f"    Done in {elapsed:.1f}s | {n_rows} stage rows")

        except Exception as e:
            error_info = {
                "alpha": alpha,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
            errors.append(error_info)
            if verbose:
                print(f"    ERROR: {e}")

    all_stages = pd.concat(all_stage_dfs, ignore_index=True) if all_stage_dfs else pd.DataFrame()
    all_iters = pd.concat(all_iter_dfs, ignore_index=True) if all_iter_dfs else pd.DataFrame()

    if verbose:
        print(f"\n{'='*60}")
        print(f"Experiment complete: {len(all_stages)} stage rows, "
              f"{len(all_iters)} iter rows, {len(errors)} errors")

    return all_stages, all_iters, errors

print("Experiment runner function defined.")


---
## 5. Run Small Sample (n=500) -- Verification

Run all 14 methods on a small subsample (n=500) to verify correctness before the full run.

In [ ]:
# Run small experiment for verification
print("=" * 60)
print(f"SMALL EXPERIMENT: n_sample={N_SAMPLE_SMALL}")
print(f"Alpha values: {ALPHA_VALUES}")
print(f"Methods: {len(method_configs)}")
print(f"Lambda values: {base_train_cfg['lambdas']}")
print(f"Seeds: {base_train_cfg['seeds']}")
print(f"Steps per lambda: {base_train_cfg['steps_per_lambda']}")
total_training_runs = (
    len(ALPHA_VALUES) * len(method_configs) *
    len(base_train_cfg['seeds']) * len(base_train_cfg['lambdas'])
)
print(f"Total training runs: {total_training_runs}")
print("=" * 60)

stage_small, iter_small, errors_small = run_full_experiment(
    n_sample=N_SAMPLE_SMALL,
    alpha_values=ALPHA_VALUES,
    method_configs=method_configs,
    base_train_cfg=base_train_cfg,
    verbose=True,
)

In [ ]:
# Print errors if any
if errors_small:
    print(f"\nEncountered {len(errors_small)} errors:")
    for err in errors_small:
        print(f"  alpha={err['alpha']}, method={err['method']}: {err['error']}")
else:
    print("No errors encountered.")

In [ ]:
# Summary table for small experiment
if len(stage_small) > 0:
    # Build aggregation dict including normalized regret if available
    agg_dict = {
        'test_regret': ['mean', 'std'],
    }
    if 'test_regret_normalized' in stage_small.columns:
        agg_dict['test_regret_normalized'] = ['mean', 'std']
    agg_dict.update({
        'test_fairness': ['mean', 'std'],
        'test_pred_mse': ['mean', 'std'],
    })

    agg_small = (
        stage_small
        .groupby(['config_name', 'alpha_fair'])
        .agg(agg_dict)
        .round(6)
    )
    print("\nSmall Experiment Results (mean +/- std across seeds and lambdas):")
    print(agg_small.to_string())
else:
    print("No results to display.")

---
## 6. Run Full Dataset

Run all 14 methods on the full dataset (n=48,784). This will take significantly longer.
The implementation uses the efficient VJP (vector-Jacobian product) which avoids forming
the full n x n Jacobian matrix, making it O(n) instead of O(n^2).

**Estimated runtime:** 2-6 hours on Colab T4 GPU depending on method.

In [ ]:
# Run full experiment
print("=" * 60)
print(f"FULL EXPERIMENT: n_sample={N_SAMPLE_FULL} (all patients)")
print(f"Alpha values: {ALPHA_VALUES}")
print(f"Methods: {len(method_configs)}")
print(f"Lambda values: {base_train_cfg['lambdas']}")
print(f"Seeds: {base_train_cfg['seeds']}")
print(f"Steps per lambda: {base_train_cfg['steps_per_lambda']}")
total_training_runs = (
    len(ALPHA_VALUES) * len(method_configs) *
    len(base_train_cfg['seeds']) * len(base_train_cfg['lambdas'])
)
print(f"Total training runs: {total_training_runs}")
print("=" * 60)

stage_full, iter_full, errors_full = run_full_experiment(
    n_sample=N_SAMPLE_FULL,
    alpha_values=ALPHA_VALUES,
    method_configs=method_configs,
    base_train_cfg=base_train_cfg,
    verbose=True,
)

In [ ]:
# Print errors if any
if errors_full:
    print(f"\nEncountered {len(errors_full)} errors:")
    for err in errors_full:
        print(f"  alpha={err['alpha']}, method={err['method']}: {err['error']}")
else:
    print("No errors encountered.")

In [ ]:
# Save results to CSV for later analysis
results_dir = RESULTS_DIR
os.makedirs(results_dir, exist_ok=True)

if len(stage_full) > 0:
    stage_path = os.path.join(results_dir, 'stage_results_full.csv')
    stage_full.to_csv(stage_path, index=False)
    print(f"Stage results saved to: {stage_path}")
    # Verify normalized regret columns are present
    norm_cols = [c for c in stage_full.columns if 'normalized' in c]
    if norm_cols:
        print(f"  Includes normalized regret columns: {norm_cols}")
    else:
        print("  WARNING: No normalized regret columns found in stage results!")

if len(iter_full) > 0:
    iter_path = os.path.join(results_dir, 'iter_logs_full.csv')
    iter_full.to_csv(iter_path, index=False)
    print(f"Iteration logs saved to: {iter_path}")

---
## 7. Results Analysis

Analyze and visualize the experiment results.

In [ ]:
# ========================================================================
# Load results — either from in-memory DataFrames or from saved CSVs
# ========================================================================
# If you ran new methods via run_methods.py and want to re-plot, set
# RELOAD_FROM_CSV = True and re-run from this cell downward.

RELOAD_FROM_CSV = False  # <-- Set True to reload from CSV after appending new methods

if RELOAD_FROM_CSV:
    # Reload from CSV (picks up any methods appended by run_methods.py)
    _stage_path = os.path.join(RESULTS_DIR, 'stage_results_full.csv')
    _iter_path  = os.path.join(RESULTS_DIR, 'iter_logs_full.csv')
    assert os.path.exists(_stage_path), f"No stage CSV at {_stage_path}"
    stage_df = pd.read_csv(_stage_path)
    iter_df  = pd.read_csv(_iter_path) if os.path.exists(_iter_path) else pd.DataFrame()
    experiment_label = f"Reloaded from CSV ({len(stage_df)} rows)"
elif len(stage_full) > 0:
    stage_df = stage_full.copy()
    iter_df = iter_full.copy()
    experiment_label = "Full Dataset"
else:
    stage_df = stage_small.copy()
    iter_df = iter_small.copy()
    experiment_label = "Small Sample (n=500)"

print(f"Analyzing results from: {experiment_label}")
print(f"Stage results: {len(stage_df)} rows")
print(f"Iteration logs: {len(iter_df)} rows")
print(f"Methods present: {sorted(stage_df['config_name'].unique().tolist())}")
print(f"Alphas present:  {sorted(stage_df['alpha_fair'].unique().tolist())}")


In [ ]:
# ========================================================================
# Method Comparison Table (with both raw and normalized regret)
# ========================================================================

if len(stage_df) > 0 and 'test_regret' in stage_df.columns:
    # Check if normalized regret columns exist
    has_norm_regret = 'test_regret_normalized' in stage_df.columns

    # Build aggregation dict
    agg_dict = {
        'regret_mean': ('test_regret', 'mean'),
        'regret_std': ('test_regret', 'std'),
    }
    if has_norm_regret:
        agg_dict['norm_regret_mean'] = ('test_regret_normalized', 'mean')
        agg_dict['norm_regret_std'] = ('test_regret_normalized', 'std')
    agg_dict.update({
        'fairness_mean': ('test_fairness', 'mean'),
        'fairness_std': ('test_fairness', 'std'),
        'pred_mse_mean': ('test_pred_mse', 'mean'),
        'pred_mse_std': ('test_pred_mse', 'std'),
        'n_runs': ('test_regret', 'count'),
    })

    summary = (
        stage_df
        .groupby(['config_name', 'alpha_fair'])
        .agg(**agg_dict)
        .reset_index()
        .sort_values(['alpha_fair', 'regret_mean'])
    )

    for alpha in ALPHA_VALUES:
        print(f"\n{'='*100}")
        print(f"Results for alpha_fair = {alpha}")
        print(f"{'='*100}")
        sub = summary[summary['alpha_fair'] == alpha].copy()

        # Display columns
        display_cols = ['config_name', 'regret_mean', 'regret_std']
        col_names = ['Method', 'Regret (mean)', 'Regret (std)']
        if has_norm_regret:
            display_cols += ['norm_regret_mean', 'norm_regret_std']
            col_names += ['NormRegret (mean)', 'NormRegret (std)']
        display_cols += ['fairness_mean', 'fairness_std',
                         'pred_mse_mean', 'pred_mse_std', 'n_runs']
        col_names += ['Fairness (mean)', 'Fairness (std)',
                      'Pred MSE (mean)', 'Pred MSE (std)', 'N']

        sub_display = sub[display_cols].copy()
        sub_display.columns = col_names
        print(sub_display.to_string(index=False, float_format='%.6f'))

    if not has_norm_regret:
        print("\nNote: Normalized regret columns not found in results. "
              "This may indicate the task does not compute normalized regret.")
else:
    print("No results available for summary table.")

In [ ]:
# ========================================================================
# Pareto Frontier Plot: Regret vs Fairness
# ========================================================================

if len(stage_df) > 0 and 'test_regret' in stage_df.columns:
    # Define method categories and colors
    non_moo_methods = {'FPTO', 'FDFL', 'FFO', 'LANCER'}
    ws_methods = {'WS-equal', 'WS-dec', 'WS-fair', 'WS-balanced'}
    gradient_moo = {'MGDA', 'PCGrad', 'PLG-FP', 'PLG-PP', 'CAGrad', 'FAMO'}

    # Color map
    color_map = {
        'FPTO': '#1f77b4', 'FDFL': '#ff7f0e', 'FFO': '#2ca02c', 'LANCER': '#d62728',
        'WS-equal': '#9467bd', 'WS-dec': '#8c564b', 'WS-fair': '#e377c2', 'WS-balanced': '#7f7f7f',
        'MGDA': '#bcbd22', 'PCGrad': '#17becf', 'PLG-FP': '#aec7e8', 'PLG-PP': '#ffbb78',
        'CAGrad': '#98df8a', 'FAMO': '#ff9896',
    }
    marker_map = {
        'FPTO': 'o', 'FDFL': 's', 'FFO': '^', 'LANCER': 'D',
        'WS-equal': 'v', 'WS-dec': '<', 'WS-fair': '>', 'WS-balanced': 'p',
        'MGDA': 'h', 'PCGrad': '*', 'PLG-FP': 'X', 'PLG-PP': 'P',
        'CAGrad': 'd', 'FAMO': 'H',
    }

    fig, axes = plt.subplots(1, len(ALPHA_VALUES), figsize=(8*len(ALPHA_VALUES), 6))
    if len(ALPHA_VALUES) == 1:
        axes = [axes]

    for ax_idx, alpha in enumerate(ALPHA_VALUES):
        ax = axes[ax_idx]
        sub = stage_df[stage_df['alpha_fair'] == alpha]

        for method_name in sub['config_name'].unique():
            msub = sub[sub['config_name'] == method_name]
            color = color_map.get(method_name, '#333333')
            marker = marker_map.get(method_name, 'o')

            # Plot individual points (each seed x lambda combination)
            ax.scatter(
                msub['test_regret'], msub['test_fairness'],
                c=color, marker=marker, s=50, alpha=0.6,
                label=method_name, edgecolors='white', linewidths=0.5,
            )

            # Plot mean with larger marker
            ax.scatter(
                msub['test_regret'].mean(), msub['test_fairness'].mean(),
                c=color, marker=marker, s=150, alpha=1.0,
                edgecolors='black', linewidths=1.5,
            )

        ax.set_xlabel('Test Regret (lower is better)', fontsize=12)
        ax.set_ylabel('Test Fairness Loss / MAD (lower is better)', fontsize=12)
        ax.set_title(f'Pareto Frontier: alpha = {alpha}', fontsize=14)
        ax.legend(fontsize=8, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'pareto_regret_vs_fairness.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print("Pareto frontier plot saved.")

In [ ]:
# ========================================================================
# Pareto Frontier Plot: NORMALIZED Regret vs Fairness
# ========================================================================

if len(stage_df) > 0 and 'test_regret_normalized' in stage_df.columns:
    fig, axes = plt.subplots(1, len(ALPHA_VALUES), figsize=(8*len(ALPHA_VALUES), 6))
    if len(ALPHA_VALUES) == 1:
        axes = [axes]

    for ax_idx, alpha in enumerate(ALPHA_VALUES):
        ax = axes[ax_idx]
        sub = stage_df[stage_df['alpha_fair'] == alpha]

        for method_name in sub['config_name'].unique():
            msub = sub[sub['config_name'] == method_name]
            color = color_map.get(method_name, '#333333')
            marker = marker_map.get(method_name, 'o')

            ax.scatter(
                msub['test_regret_normalized'], msub['test_fairness'],
                c=color, marker=marker, s=50, alpha=0.6,
                label=method_name, edgecolors='white', linewidths=0.5,
            )
            ax.scatter(
                msub['test_regret_normalized'].mean(), msub['test_fairness'].mean(),
                c=color, marker=marker, s=150, alpha=1.0,
                edgecolors='black', linewidths=1.5,
            )

        ax.set_xlabel('Normalized Test Regret (lower is better)', fontsize=12)
        ax.set_ylabel('Test Fairness Loss / MAD (lower is better)', fontsize=12)
        ax.set_title(f'Pareto Frontier (Normalized Regret): alpha = {alpha}', fontsize=14)
        ax.legend(fontsize=8, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'pareto_norm_regret_vs_fairness.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print("Normalized regret Pareto frontier plot saved.")
else:
    print("Normalized regret column not available; skipping normalized Pareto plot.")

In [ ]:
# ========================================================================
# Pareto Frontier Plot: Regret vs Prediction MSE
# ========================================================================

if len(stage_df) > 0 and 'test_regret' in stage_df.columns:
    fig, axes = plt.subplots(1, len(ALPHA_VALUES), figsize=(8*len(ALPHA_VALUES), 6))
    if len(ALPHA_VALUES) == 1:
        axes = [axes]

    for ax_idx, alpha in enumerate(ALPHA_VALUES):
        ax = axes[ax_idx]
        sub = stage_df[stage_df['alpha_fair'] == alpha]

        for method_name in sub['config_name'].unique():
            msub = sub[sub['config_name'] == method_name]
            color = color_map.get(method_name, '#333333')
            marker = marker_map.get(method_name, 'o')

            ax.scatter(
                msub['test_regret'], msub['test_pred_mse'],
                c=color, marker=marker, s=50, alpha=0.6,
                label=method_name, edgecolors='white', linewidths=0.5,
            )
            ax.scatter(
                msub['test_regret'].mean(), msub['test_pred_mse'].mean(),
                c=color, marker=marker, s=150, alpha=1.0,
                edgecolors='black', linewidths=1.5,
            )

        ax.set_xlabel('Test Regret (lower is better)', fontsize=12)
        ax.set_ylabel('Test Prediction MSE (lower is better)', fontsize=12)
        ax.set_title(f'Regret vs Prediction MSE: alpha = {alpha}', fontsize=14)
        ax.legend(fontsize=8, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'pareto_regret_vs_mse.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print("Regret vs MSE plot saved.")

In [ ]:
# ========================================================================
# 3D Pareto Plot: Regret vs Fairness vs MSE
# ========================================================================

if len(stage_df) > 0 and 'test_regret' in stage_df.columns:
    from mpl_toolkits.mplot3d import Axes3D

    fig = plt.figure(figsize=(8*len(ALPHA_VALUES), 7))

    for ax_idx, alpha in enumerate(ALPHA_VALUES):
        ax = fig.add_subplot(1, len(ALPHA_VALUES), ax_idx + 1, projection='3d')
        sub = stage_df[stage_df['alpha_fair'] == alpha]

        for method_name in sub['config_name'].unique():
            msub = sub[sub['config_name'] == method_name]
            color = color_map.get(method_name, '#333333')
            marker = marker_map.get(method_name, 'o')

            ax.scatter(
                msub['test_regret'], msub['test_fairness'], msub['test_pred_mse'],
                c=color, marker=marker, s=40, alpha=0.7, label=method_name,
            )

        ax.set_xlabel('Regret', fontsize=9)
        ax.set_ylabel('Fairness (MAD)', fontsize=9)
        ax.set_zlabel('Pred MSE', fontsize=9)
        ax.set_title(f'3D Pareto: alpha = {alpha}', fontsize=12)
        ax.legend(fontsize=6, loc='upper left', ncol=2)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'pareto_3d.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print("3D Pareto plot saved.")

In [ ]:
# ========================================================================
# Per-Lambda Comparison: how fairness penalty affects each method
# ========================================================================

if len(stage_df) > 0 and 'lambda' in stage_df.columns:
    has_norm = 'test_regret_normalized' in stage_df.columns
    n_cols = 4 if has_norm else 3
    fig, axes = plt.subplots(len(ALPHA_VALUES), n_cols, figsize=(6*n_cols, 5*len(ALPHA_VALUES)))
    if len(ALPHA_VALUES) == 1:
        axes = axes.reshape(1, -1)

    metrics = [('test_regret', 'Regret'), ('test_fairness', 'Fairness (MAD)'),
               ('test_pred_mse', 'Prediction MSE')]
    if has_norm:
        metrics.insert(1, ('test_regret_normalized', 'Normalized Regret'))

    for row_idx, alpha in enumerate(ALPHA_VALUES):
        sub = stage_df[stage_df['alpha_fair'] == alpha]

        for col_idx, (metric, metric_label) in enumerate(metrics):
            ax = axes[row_idx, col_idx]

            # Group by method and lambda, compute mean across seeds
            grouped = (
                sub.groupby(['config_name', 'lambda'])[metric]
                .agg(['mean', 'std'])
                .reset_index()
            )

            for method_name in grouped['config_name'].unique():
                msub = grouped[grouped['config_name'] == method_name]
                color = color_map.get(method_name, '#333333')
                ax.plot(msub['lambda'], msub['mean'], '-o',
                       color=color, label=method_name, markersize=4, linewidth=1.5)
                ax.fill_between(
                    msub['lambda'],
                    msub['mean'] - msub['std'],
                    msub['mean'] + msub['std'],
                    color=color, alpha=0.15,
                )

            ax.set_xlabel('Lambda (fairness weight)', fontsize=10)
            ax.set_ylabel(metric_label, fontsize=10)
            ax.set_title(f'{metric_label} vs Lambda (alpha={alpha})', fontsize=11)
            ax.legend(fontsize=6, ncol=3)
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'lambda_sweep.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print("Lambda sweep plot saved.")

In [ ]:
# ========================================================================
# Training Curves: Loss over iterations
# ========================================================================

if len(iter_df) > 0 and 'loss_dec' in iter_df.columns:
    # Pick alpha=2.0 (or first available), lambda=0.2 (or first available)
    plot_alpha = 2.0 if 2.0 in iter_df['alpha_fair'].unique() else iter_df['alpha_fair'].iloc[0]
    sub_iter = iter_df[iter_df['alpha_fair'] == plot_alpha]

    # Pick a specific lambda stage
    if 'lambda' in sub_iter.columns:
        available_lambdas = sorted(sub_iter['lambda'].unique())
        plot_lambda = 0.2 if 0.2 in available_lambdas else available_lambdas[-1]
        sub_iter = sub_iter[sub_iter['lambda'] == plot_lambda]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    loss_cols = [('loss_dec', 'Decision Regret'), ('loss_pred', 'Prediction Loss'),
                 ('loss_fair', 'Fairness Loss')]

    for ax_idx, (col, label) in enumerate(loss_cols):
        ax = axes[ax_idx]
        if col not in sub_iter.columns:
            continue

        for method_name in sub_iter['config_name'].unique():
            msub = sub_iter[sub_iter['config_name'] == method_name]
            grouped = msub.groupby('iter')[col].agg(['mean', 'std'])
            color = color_map.get(method_name, '#333333')

            ax.plot(grouped.index, grouped['mean'], color=color,
                   label=method_name, linewidth=1.2)
            ax.fill_between(
                grouped.index,
                (grouped['mean'] - grouped['std']).clip(lower=0),
                grouped['mean'] + grouped['std'],
                color=color, alpha=0.15,
            )

        ax.set_xlabel('Iteration', fontsize=10)
        ax.set_ylabel(label, fontsize=10)
        ax.set_title(f'{label} (alpha={plot_alpha}, lambda={plot_lambda})', fontsize=11)
        ax.legend(fontsize=6, ncol=3)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'training_curves.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print("Training curves plot saved.")

In [ ]:
# ========================================================================
# Gradient Conflict Analysis
# ========================================================================

if len(iter_df) > 0 and 'cos_dec_pred' in iter_df.columns:
    plot_alpha = 2.0 if 2.0 in iter_df['alpha_fair'].unique() else iter_df['alpha_fair'].iloc[0]
    sub_iter = iter_df[iter_df['alpha_fair'] == plot_alpha]

    if 'lambda' in sub_iter.columns:
        available_lambdas = sorted(sub_iter['lambda'].unique())
        plot_lambda = 0.2 if 0.2 in available_lambdas else available_lambdas[-1]
        sub_iter = sub_iter[sub_iter['lambda'] == plot_lambda]

    cos_pairs = [
        ('cos_dec_pred', 'cos(g_dec, g_pred)'),
        ('cos_dec_fair', 'cos(g_dec, g_fair)'),
        ('cos_pred_fair', 'cos(g_pred, g_fair)'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax_idx, (col, label) in enumerate(cos_pairs):
        ax = axes[ax_idx]
        if col not in sub_iter.columns:
            continue

        for method_name in sub_iter['config_name'].unique():
            msub = sub_iter[sub_iter['config_name'] == method_name]
            grouped = msub.groupby('iter')[col].mean()
            color = color_map.get(method_name, '#333333')
            ax.plot(grouped.index, grouped.values, color=color,
                   label=method_name, linewidth=1.2)

        ax.axhline(y=0, color='black', linewidth=0.5, linestyle=':')
        ax.set_xlabel('Iteration', fontsize=10)
        ax.set_ylabel(label, fontsize=10)
        ax.set_title(f'{label} (alpha={plot_alpha})', fontsize=11)
        ax.set_ylim(-1.1, 1.1)
        ax.legend(fontsize=6, ncol=3)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'gradient_conflict.png'), dpi=150,
                bbox_inches='tight')
    plt.show()
    print("Gradient conflict plot saved.")

In [ ]:
# ========================================================================
# Per-Alpha Comparison: Bar chart of best-lambda results
# ========================================================================

if len(stage_df) > 0 and 'test_regret' in stage_df.columns:
    has_norm = 'test_regret_normalized' in stage_df.columns

    # For each method and alpha, select the lambda that minimizes
    # a combined score: regret + fairness (Pareto-proxy)
    best_rows = []
    for alpha in ALPHA_VALUES:
        sub = stage_df[stage_df['alpha_fair'] == alpha]
        for method_name in sub['config_name'].unique():
            msub = sub[sub['config_name'] == method_name]
            # Average across seeds for each lambda
            agg_cols = {
                'regret': ('test_regret', 'mean'),
                'fairness': ('test_fairness', 'mean'),
                'pred_mse': ('test_pred_mse', 'mean'),
            }
            if has_norm:
                agg_cols['norm_regret'] = ('test_regret_normalized', 'mean')
            avg = msub.groupby('lambda').agg(**agg_cols).reset_index()
            # Normalize and pick best combined
            if len(avg) > 0:
                avg['combined'] = avg['regret'] / max(avg['regret'].max(), 1e-12) + \
                                  avg['fairness'] / max(avg['fairness'].max(), 1e-12)
                best = avg.loc[avg['combined'].idxmin()]
                row = {
                    'alpha': alpha,
                    'method': method_name,
                    'best_lambda': best['lambda'],
                    'regret': best['regret'],
                    'fairness': best['fairness'],
                    'pred_mse': best['pred_mse'],
                }
                if has_norm:
                    row['norm_regret'] = best['norm_regret']
                best_rows.append(row)

    best_df = pd.DataFrame(best_rows)

    if len(best_df) > 0:
        n_bar_plots = 4 if has_norm else 3
        fig, axes = plt.subplots(1, n_bar_plots, figsize=(6*n_bar_plots, 6))
        bar_metrics = [('regret', 'Best Regret')]
        if has_norm:
            bar_metrics.append(('norm_regret', 'Best Normalized Regret'))
        bar_metrics += [('fairness', 'Best Fairness (MAD)'),
                        ('pred_mse', 'Best Pred MSE')]

        for ax_idx, (metric, label) in enumerate(bar_metrics):
            ax = axes[ax_idx]
            width = 0.35
            methods_list = sorted(best_df['method'].unique())
            x = np.arange(len(methods_list))

            for i, alpha in enumerate(ALPHA_VALUES):
                sub = best_df[best_df['alpha'] == alpha]
                vals = []
                for m in methods_list:
                    row = sub[sub['method'] == m]
                    vals.append(row[metric].values[0] if len(row) > 0 else 0)
                offset = (i - 0.5) * width
                ax.bar(x + offset, vals, width, label=f'alpha={alpha}', alpha=0.8)

            ax.set_xticks(x)
            ax.set_xticklabels(methods_list, rotation=45, ha='right', fontsize=8)
            ax.set_ylabel(label, fontsize=10)
            ax.set_title(label, fontsize=12)
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.savefig(os.path.join(results_dir, 'per_alpha_comparison.png'), dpi=150,
                    bbox_inches='tight')
        plt.show()
        print("Per-alpha comparison plot saved.")

In [ ]:
# ========================================================================
# Final Summary
# ========================================================================

print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Experiment type: {experiment_label}")
print(f"Total stage results: {len(stage_df)}")
print(f"Total iteration logs: {len(iter_df)}")
print(f"Alpha values tested: {ALPHA_VALUES}")
print(f"Methods tested: {sorted(stage_df['config_name'].unique().tolist()) if len(stage_df) > 0 else 'none'}")
print(f"Results saved to: {results_dir}")

# Check for key columns
if len(stage_df) > 0:
    has_norm = 'test_regret_normalized' in stage_df.columns
    print(f"Normalized regret available: {has_norm}")
    if has_norm:
        print(f"  Columns: test_regret_normalized, test_regret_normalized_pred_obj")
print("=" * 70)

---
## Appendix: Complete Guide to Results & Metrics

This section explains **every column** in the two output CSVs and how to interpret the experiment as a whole. Intended as a self-contained reference when sharing results.

---

### A. The Experiment at a Glance

We train a neural predictor $\hat{r} = f_\theta(x)$ that estimates patient benefit from features. A downstream solver uses $\hat{r}$ to allocate scarce medical resources under a budget constraint. The pipeline evaluates three competing objectives:

| Objective | Loss name | Formula | Goal |
|-----------|-----------|---------|------|
| **Decision quality** | `regret` | $W(d^*_{\text{true}}) - W(d^*_{\text{pred}})$ | Minimize welfare lost from using predictions instead of ground truth |
| **Prediction accuracy** | `pred_mse` | $\frac{1}{n}\sum_i (\hat{r}_i - r_i)^2$ | Minimize prediction error |
| **Demographic fairness** | `fairness` | Smoothed MAD of per-group MSE | Equalize prediction quality across racial groups |

$W(\cdot)$ is the $\alpha$-fair social welfare function. $d^*_{\text{true}}$ and $d^*_{\text{pred}}$ are optimal allocations under true vs. predicted benefits.

---

### B. Normalized Regret

Raw regret values are on different scales for different $\alpha_{\text{fair}}$ values (the utility function changes shape). We normalize:

$$\text{NormRegret} = \frac{W(d^*_{\text{true}}) - W(d^*_{\text{pred}})}{|W(d^*_{\text{true}})| + \epsilon}$$

A value of **0.05 means we lose 5% of the optimal welfare**. This is comparable across $\alpha$ settings.

There is a second variant `regret_normalized_pred_obj` that divides by $|W(d^*_{\text{pred}})|$ instead — useful as a sanity check but less standard.

---

### C. Index Columns (both CSVs)

These columns identify *which* run a row belongs to.

| Column | Meaning |
|--------|---------|
| `task` | Always `medical_resource_allocation` in this experiment. |
| `method` | The backend method code: `fpto`, `fdfl`, `fplg`, or `ffo`. Note: MOO methods (MGDA, PCGrad, WS-*) all run through `fplg`. |
| `config_name` | Human-readable name: `FPTO`, `FDFL`, `FFO`, `WS-equal`, `WS-dec`, `MGDA`, `PCGrad`. **Use this to distinguish methods.** |
| `seed` | Random seed (we use 3 seeds: 11, 22, 33 for variance estimation). |
| `stage_idx` | Which lambda value (0, 1, 2, 3 corresponding to the lambda list). |
| `lambda` | Fairness penalty weight: 0.0, 0.05, 0.2, 0.5. Higher = more fairness pressure. |
| `alpha_fair` | The $\alpha$ parameter of the fairness-aware utility function. Each $\alpha$ defines a different problem. |
| `predictor_family` | Model architecture used (e.g. `mlp_2x64_softplus`). |
| `device` | Compute device (`cuda` or `cpu`). |

---

### D. `stage_results_full.csv` — One Row Per (method, seed, lambda)

This is the **primary results table**. Each row is the final evaluation after training completes for one (method, seed, lambda) configuration.

#### D.1 Primary Metrics (what you report)

| Column | What it is | Lower is better? |
|--------|-----------|-------------------|
| `test_regret` | Decision regret on test set (raw) | ✅ Yes |
| `test_regret_normalized` | Normalized regret: regret / \|optimal welfare\| | ✅ Yes |
| `test_fairness` | Fairness violation on test set | ✅ Yes |
| `test_pred_mse` | Prediction MSE on test set | ✅ Yes |
| `val_regret` | Same metrics on validation set | ✅ Yes |
| `val_regret_normalized` | | ✅ Yes |
| `val_fairness` | | ✅ Yes |
| `val_pred_mse` | | ✅ Yes |
| `test_regret_normalized_pred_obj` | Regret normalized by predicted-decision objective | ✅ Yes |
| `val_regret_normalized_pred_obj` | (same, on val) | ✅ Yes |

#### D.2 Training Diagnostics

| Column | What it is | How to read it |
|--------|-----------|----------------|
| `stage_wallclock_sec` | Wall-clock seconds for this training stage | Method speed comparison |
| `cumulative_wallclock_sec` | Cumulative time across all lambda stages for this (method, seed) | Total training cost |
| `nan_or_inf_steps` / `nan_steps` | Number of training steps skipped due to NaN/Inf gradients | Should be 0; if high, method is unstable |
| `exploding_steps` | Steps where gradient norm exceeded `explode_threshold` | Should be 0 |
| `weight_norm` | L2 norm of model parameters after training | Sanity check for parameter growth |

#### D.3 Gradient Statistics (averaged over training)

| Column | What it is |
|--------|-----------|
| `avg_grad_norm_combined` | Mean norm of the final combined gradient |
| `grad_norm_min` / `grad_norm_median` / `grad_norm_max` | Distribution of combined gradient norms |
| `avg_cos_dec_pred` | Mean cosine similarity between decision and prediction gradients |
| `avg_cos_dec_fair` | Mean cosine between decision and fairness gradients |
| `avg_cos_pred_fair` | Mean cosine between prediction and fairness gradients |

**Interpretation**: Negative cosine values mean the two objectives push in opposite directions (conflict). This is where MOO methods help — they resolve conflicts instead of blindly summing.

#### D.4 Solver / Timing

| Column | What it is |
|--------|-----------|
| `solver_calls_train` | Number of CVXPY solves during training |
| `solver_calls_eval` | Number of CVXPY solves during val/test evaluation |
| `decision_ms_train` | Milliseconds spent in the decision solver during training |
| `decision_ms_eval` | Milliseconds spent in the decision solver during evaluation |
| `solver_calls_fd_train` | Finite-difference solver calls (0 for analytical gradient methods) |
| `decision_ms_fd_train` | Time for finite-difference solves (0 for analytical methods) |
| `ffo_backward_ms` | Average backward-pass time for FFO layer (0 for non-FFO methods) |

#### D.5 Unused / Method-Specific

| Column | Note |
|--------|------|
| `nce_pool_size` | Always 0 (NCE method not in current config) |
| `lancer_surrogate_loss` | Always 0 (LANCER method not in current config) |

---

### E. `iter_logs_full.csv` — One Row Per Training Iteration (logged every `log_every` steps)

This is the **training curve data**. Use it to plot how losses evolve during training, diagnose convergence issues, and analyze gradient dynamics.

#### E.1 Iteration Index

| Column | Meaning |
|--------|---------|
| `iter` | Training iteration number within the current lambda stage |
| `alpha_t` | Current value of the prediction-loss weight schedule at this step |
| `beta_t` | Current fairness penalty weight (= `lambda` for penalty mode) |
| `lr_t` | Current learning rate (after decay) |

#### E.2 Loss Values (at this iteration)

| Column | What it is |
|--------|-----------|
| `loss_dec` | Decision regret this batch (= raw regret, not normalized) |
| `loss_dec_normalized` | Normalized regret this batch |
| `loss_dec_normalized_pred_obj` | Regret normalized by predicted objective |
| `loss_pred` | Prediction MSE this batch |
| `loss_fair` | Fairness loss this batch |

**Use these to plot training curves**: `loss_dec` should decrease over iterations for decision-aware methods.

#### E.3 Per-Objective Gradient Norms

| Column | What it is |
|--------|-----------|
| `grad_norm_dec` | $\|\nabla_\theta L_{\text{dec}}\|$ — decision gradient magnitude |
| `grad_norm_pred` | $\|\nabla_\theta L_{\text{pred}}\|$ — prediction gradient magnitude |
| `grad_norm_fair` | $\|\nabla_\theta L_{\text{fair}}\|$ — fairness gradient magnitude |
| `grad_norm_combined` | $\|\nabla_\theta L_{\text{combined}}\|$ — final gradient after combination |

**Interpretation**: If one gradient norm dominates (e.g. `grad_norm_dec` >> others), that objective drives training. MOO methods aim to balance these.

#### E.4 Gradient Conflict Indicators

| Column | What it is | Key threshold |
|--------|-----------|---------------|
| `cos_dec_pred` | $\cos(\nabla L_{\text{dec}}, \nabla L_{\text{pred}})$ | < 0 means conflict |
| `cos_dec_fair` | $\cos(\nabla L_{\text{dec}}, \nabla L_{\text{fair}})$ | < 0 means conflict |
| `cos_pred_fair` | $\cos(\nabla L_{\text{pred}}, \nabla L_{\text{fair}})$ | < 0 means conflict |

**These are the most important diagnostic columns.** If decision and fairness gradients consistently conflict (negative cosine), naive gradient summing hurts — MOO methods handle this.

#### E.5 Guided Merge Diagnostics (non-MOO methods only)

These describe how the alpha-scheduled prediction-loss-guided merge works internally:

| Column | What it is |
|--------|-----------|
| `guided_norm_dec` | Norm of decision gradient before merge |
| `guided_norm_pred` | Norm of prediction gradient before merge |
| `guided_scale` | Scale factor applied to the merged direction |
| `guided_ratio_pred_over_dec` | Ratio of prediction to decision gradient norms |
| `guided_dir_norm` | Norm of the merged direction vector |
| `delta_theta_l2` | L2 norm of the parameter update $\|\theta_{t+1} - \theta_t\|$ |

#### E.6 MOO Handler Diagnostics (only populated for MOO methods)

| Column | Populated by | What it is |
|--------|-------------|-----------|
| `mo_grad_norm_decision_regret` | All MOO | Per-objective gradient norm (param-space) |
| `mo_grad_norm_pred_fairness` | All MOO | |
| `mo_grad_norm_pred_loss` | All MOO | |
| `mo_cos_decision_regret_pred_fairness` | All MOO | Pairwise cosine in param-space |
| `mo_cos_decision_regret_pred_loss` | All MOO | |
| `mo_cos_pred_fairness_pred_loss` | All MOO | |
| `direction_alignment_with_dec_regret` | All MOO | $\cos(\text{combined direction}, \nabla L_{\text{dec}})$ — how much the final direction aligns with decision quality |
| `stationarity_proxy` | All MOO | $\min_{\lambda \in \Delta} \|\sum \lambda_i g_i\|$ — how close to Pareto stationary |
| `mo_ws_weight_*` | WS only | Normalized weights used for each objective |
| `mo_mgda_lambda_*` | MGDA only | Convex hull weights found by the QP solver |
| `mo_mgda_min_norm` | MGDA only | Minimum-norm value (= stationarity measure) |
| `mo_pcgrad_n_projections` | PCGrad only | Number of conflict projections performed |
| `mo_pcgrad_conflict_fraction` | PCGrad only | Fraction of gradient pairs that conflicted |
| `mo_method` | All MOO | String identifier (`weighted_sum`, `mgda`, `pcgrad`) |
| `grad_backend` | All | Gradient computation method (`analytic`) |

#### E.7 Stability Flags

| Column | What it is |
|--------|-----------|
| `nan_or_inf_flag` | 1 if this iteration had NaN/Inf (step was skipped) |
| `nan_or_inf_steps_so_far` | Cumulative NaN steps up to this iteration |
| `exploding_steps_so_far` | Cumulative exploding gradient steps |
| `solver_calls` | CVXPY solves this iteration |
| `decision_ms` | Decision solver time this iteration (ms) |

---

### F. How to Read the Summary Table

The summary table at the top of results averages across **all seeds (3) and all lambdas (4)**, giving N=12 per method per alpha.

**Caveat**: Different lambdas represent fundamentally different fairness-regret tradeoff points. Averaging across them mixes a "no fairness" run (lambda=0) with a "heavy fairness" run (lambda=0.5). The Pareto frontier plots give a cleaner picture.

**For a quick one-line comparison**: Use `NormRegret (mean)` — it is comparable across alpha values and gives the fraction of optimal welfare lost.

**For a thorough comparison**: Look at the Pareto frontier (regret vs fairness), where each dot is one (method, lambda) and the lower-left corner is ideal.

---

